# Victorian Rental Rights RAG Baseline

This notebook builds the initial RAG pipeline using the Version 1 rental-rights knowledge base and test collection. The baseline will first evaluate retrieval performance before adding answer generation and faithfulness evaluation.

## Notebook Summary: Baseline Retrieval


We first loaded the Version 1 knowledge base and the 15-question test collection, then built a simple BM25 retriever over the 32 rental-rights chunks. BM25 was used as the baseline because it provides a straightforward keyword-based retrieval method that we can later compare against more advanced approaches.

Retrieval was evaluated using Recall, MRR and NDCG so we could measure not only whether the correct evidence was found, but also how highly it was ranked. We also compared different retrieval depths (`k = 1, 3, 5, 10`) to see how much context was needed to cover the relevant evidence.

The results showed that `top_k = 5` provided a good balance. Mean Recall increased substantially from 0.806 at `k = 3` to 0.979 at `k = 5`, while increasing to `k = 10` only produced a small further improvement. For this reason, five retrieved chunks were selected for the baseline RAG pipeline before moving into answer generation.

In [1]:
from pathlib import Path
import pandas as pd

CURRENT_DIR = Path.cwd()

if (CURRENT_DIR / "data" / "processed").exists():
    DATA_DIR = CURRENT_DIR / "data" / "processed"
elif (CURRENT_DIR.parent / "data" / "processed").exists():
    DATA_DIR = CURRENT_DIR.parent / "data" / "processed"
else:
    raise FileNotFoundError("Could not locate data/processed folder")

kb_df = pd.read_json(
    DATA_DIR / "rental_kb_chunks.jsonl",
    lines=True
)

test_df = pd.read_json(
    DATA_DIR / "rental_test_collection_v1.jsonl",
    lines=True
)

print("Knowledge-base chunks:", len(kb_df))
print("Test questions:", len(test_df))

display(kb_df.head())
display(test_df.head())

Knowledge-base chunks: 32
Test questions: 15


,chunk_id,document_id,document_name,page,text,source_file
0,RG_P04,RG,Renters Guide,4,Introduction\nVictoria has some of the stronge...,Renters Guide.pdf
1,RG_P07,RG,Renters Guide,7,Before you apply\nDocuments and information yo...,Renters Guide.pdf
2,RG_P08,RG,Renters Guide,8,More information: consumer.vic.gov.au/unlawful...,Renters Guide.pdf
3,RG_P09,RG,Renters Guide,9,Read through and complete the rental applicati...,Renters Guide.pdf
4,RG_P11,RG,Renters Guide,11,Communicating with your rental provider\nYou c...,Renters Guide.pdf


,question_id,question,question_type,relevant_chunk_ids,expected_answer
0,K01,Can a rental provider ask me about disputes wi...,Known,[RG_P07],No. A rental provider or agent cannot ask whet...
1,K02,Can a rental provider accept an offer above th...,Known,[RG_P08],No. Rental providers and agents cannot ask for...
2,K03,Does a Victorian rental property need to have ...,Known,"[RG_P13, MS_P01]",Yes. Rental properties must have a fixed heate...
3,K04,How long do I have to return my completed cond...,Known,"[RG_P17, RG_P19]",You must return one completed and signed copy ...
4,K05,How much rent can I be asked to pay in advance?,Known,[RG_P21],"If rent is paid weekly, you can be asked for u..."


## Building the BM25 Retriever

Create a simple keyword-based BM25 index over the Version 1 knowledge base. This provides the initial retrieval baseline that later approaches can be compared against.

In [2]:
import re
from rank_bm25 import BM25Okapi

# Simple tokenizer for the baseline
def tokenize(text):
    return re.findall(r"\b\w+\b", text.lower())

# Tokenise all KB chunks and build the BM25 index
tokenized_corpus = kb_df["text"].apply(tokenize).tolist()
bm25 = BM25Okapi(tokenized_corpus)

def retrieve_bm25(question, top_k=5):
    query_tokens = tokenize(question)
    scores = bm25.get_scores(query_tokens)

    results = kb_df.copy()
    results["bm25_score"] = scores

    return (
        results
        .sort_values("bm25_score", ascending=False)
        .head(top_k)[
            ["chunk_id", "document_name", "page", "bm25_score", "text"]
        ]
        .reset_index(drop=True)
    )

# Test the retriever on the first Known question
test_question = test_df.loc[0, "question"]

print("QUESTION:")
print(test_question)

print("\nEXPECTED RELEVANT CHUNKS:")
print(test_df.loc[0, "relevant_chunk_ids"])

display(retrieve_bm25(test_question, top_k=5))

QUESTION:
Can a rental provider ask me about disputes with a previous landlord when I apply?

EXPECTED RELEVANT CHUNKS:
['RG_P07']


,chunk_id,document_name,page,bm25_score,text
0,RG_P35,Renters Guide,35,14.028242,Bond claims\nWhen a rental provider can claim ...
1,RG_P38,Renters Guide,38,13.499890,Resolving rental disputes\nHow to resolve disp...
2,RG_P04,Renters Guide,4,12.238835,Introduction\nVictoria has some of the stronge...
3,RG_P23,Renters Guide,23,11.999156,Request for rental assessment\nRequest for ren...
4,RG_P07,Renters Guide,7,11.719984,Before you apply\nDocuments and information yo...


## Evaluate Baseline Retrieval

Evaluate BM25 across the Known and Inferred questions using Recall@5, MRR@5 and NDCG@5.

In [3]:
import math

def evaluate_retrieval(row, top_k=5):
    results = retrieve_bm25(row["question"], top_k=top_k)

    retrieved_ids = results["chunk_id"].tolist()
    relevant_ids = set(row["relevant_chunk_ids"])

    # Relevant chunks retrieved
    matched = [
        chunk_id for chunk_id in retrieved_ids
        if chunk_id in relevant_ids
    ]

    recall = len(matched) / len(relevant_ids)

    # Rank of first relevant result
    first_relevant_rank = None

    for rank, chunk_id in enumerate(retrieved_ids, start=1):
        if chunk_id in relevant_ids:
            first_relevant_rank = rank
            break

    mrr = (
        1 / first_relevant_rank
        if first_relevant_rank is not None
        else 0
    )

    # Binary NDCG
    dcg = 0

    for rank, chunk_id in enumerate(retrieved_ids, start=1):
        if chunk_id in relevant_ids:
            dcg += 1 / math.log2(rank + 1)

    ideal_relevant = min(len(relevant_ids), top_k)

    idcg = sum(
        1 / math.log2(rank + 1)
        for rank in range(1, ideal_relevant + 1)
    )

    ndcg = dcg / idcg if idcg > 0 else 0

    return {
        "retrieved_chunk_ids": retrieved_ids,
        "first_relevant_rank": first_relevant_rank,
        "recall_at_5": recall,
        "mrr_at_5": mrr,
        "ndcg_at_5": ndcg
    }


# Evaluate only questions with known relevant chunks
retrieval_eval_df = test_df[
    test_df["question_type"].isin(["Known", "Inferred"])
].copy()

metrics = retrieval_eval_df.apply(
    evaluate_retrieval,
    axis=1,
    result_type="expand"
)

retrieval_eval_df = pd.concat(
    [retrieval_eval_df.reset_index(drop=True), metrics.reset_index(drop=True)],
    axis=1
)

display(
    retrieval_eval_df[
        [
            "question_id",
            "question_type",
            "relevant_chunk_ids",
            "retrieved_chunk_ids",
            "first_relevant_rank",
            "recall_at_5",
            "mrr_at_5",
            "ndcg_at_5"
        ]
    ]
)

,question_id,question_type,relevant_chunk_ids,retrieved_chunk_ids,first_relevant_rank,recall_at_5,mrr_at_5,ndcg_at_5
0,K01,Known,[RG_P07],"[RG_P35, RG_P38, RG_P04, RG_P23, RG_P07]",5,1.00,0.2,0.386853
1,K02,Known,[RG_P08],"[RG_P08, RG_P21, MS_P01, RG_P12, RG_P13]",1,1.00,1.0,1.000000
2,K03,Known,"[RG_P13, MS_P01]","[RG_P13, MS_P01, RG_P39, RG_P17, RG_P04]",1,1.00,1.0,1.000000
3,K04,Known,"[RG_P17, RG_P19]","[RG_P23, RG_P17, RG_P19, RG_P34, RG_P36]",2,1.00,0.5,0.693426
4,K05,Known,[RG_P21],"[RG_P21, RG_P22, RG_P17, RG_P23, RG_P08]",1,1.00,1.0,1.000000
5,K06,Known,[RG_P22],"[RG_P23, RG_P22, RG_P17, RG_P24, RG_P18]",2,1.00,0.5,0.630930
6,K07,Known,[RG_P24],"[RG_P24, RG_P25, RG_P26, RG_P15, MS_P01]",1,1.00,1.0,1.000000
7,K08,Known,[RG_P28],"[RG_P28, RG_P23, RG_P22, RG_P29, RG_P34]",1,1.00,1.0,1.000000
8,I01,Inferred,"[RG_P13, MS_P01, RG_P24, RG_P25]","[RG_P25, RG_P26, MS_P01, RG_P23, RG_P13]",1,0.75,1.0,0.736590
9,I02,Inferred,"[RG_P22, RG_P23]","[RG_P23, RG_P22, RG_P17, RG_P18, RG_P25]",1,1.00,1.0,1.000000


## Summarise Baseline Retrieval Performance

Summarise retrieval performance overall and by question type to establish the Version 1 BM25 baseline.

In [4]:
# Summarise retrieval performance

summary_by_type = (
    retrieval_eval_df
    .groupby("question_type")[
        ["recall_at_5", "mrr_at_5", "ndcg_at_5"]
    ]
    .mean()
    .round(3)
)

overall_summary = pd.DataFrame({
    "recall_at_5": [retrieval_eval_df["recall_at_5"].mean()],
    "mrr_at_5": [retrieval_eval_df["mrr_at_5"].mean()],
    "ndcg_at_5": [retrieval_eval_df["ndcg_at_5"].mean()]
}, index=["Overall"]).round(3)

display(summary_by_type)
display(overall_summary)

,recall_at_5,mrr_at_5,ndcg_at_5
question_type,,,
Inferred,0.938,0.875,0.817
Known,1.000,0.775,0.839


,recall_at_5,mrr_at_5,ndcg_at_5
Overall,0.979,0.808,0.831


## Compare Retrieval Depth

Compare retrieval performance at different values of k to understand how much context is needed to retrieve the required evidence.

In [5]:
# Compare retrieval performance at different top-k values

def calculate_metrics(row, top_k):
    results = retrieve_bm25(row["question"], top_k=top_k)

    retrieved_ids = results["chunk_id"].tolist()
    relevant_ids = set(row["relevant_chunk_ids"])

    matched = [
        chunk_id for chunk_id in retrieved_ids
        if chunk_id in relevant_ids
    ]

    recall = len(matched) / len(relevant_ids)

    first_relevant_rank = next(
        (
            rank
            for rank, chunk_id in enumerate(retrieved_ids, start=1)
            if chunk_id in relevant_ids
        ),
        None
    )

    mrr = 1 / first_relevant_rank if first_relevant_rank else 0

    dcg = sum(
        1 / math.log2(rank + 1)
        for rank, chunk_id in enumerate(retrieved_ids, start=1)
        if chunk_id in relevant_ids
    )

    ideal_relevant = min(len(relevant_ids), top_k)

    idcg = sum(
        1 / math.log2(rank + 1)
        for rank in range(1, ideal_relevant + 1)
    )

    ndcg = dcg / idcg if idcg else 0

    return recall, mrr, ndcg


top_k_results = []

for k in [1, 3, 5, 10]:
    recalls = []
    mrrs = []
    ndcgs = []

    for _, row in retrieval_eval_df.iterrows():
        recall, mrr, ndcg = calculate_metrics(row, k)

        recalls.append(recall)
        mrrs.append(mrr)
        ndcgs.append(ndcg)

    top_k_results.append({
        "top_k": k,
        "mean_recall": sum(recalls) / len(recalls),
        "mean_mrr": sum(mrrs) / len(mrrs),
        "mean_ndcg": sum(ndcgs) / len(ndcgs)
    })

top_k_df = pd.DataFrame(top_k_results).round(3)

display(top_k_df)

,top_k,mean_recall,mean_mrr,mean_ndcg
0,1,0.465,0.667,0.667
1,3,0.806,0.792,0.760
2,5,0.979,0.808,0.831
3,10,1.000,0.808,0.843


### Baseline Retrieval Choice

Retrieval performance improved substantially when increasing from 3 to 5 retrieved chunks, with mean Recall increasing from 0.806 to 0.979. Increasing to 10 chunks produced only a small additional improvement, while doubling the context passed to the generation model. For the baseline RAG pipeline, `top_k = 5` was therefore selected as a balance between evidence coverage and unnecessary context.

## Prepare Retrieved Context for Generation

Format the top five BM25 results into a consistent context block for the language model. Source identifiers are kept with each chunk so generated claims and citations can later be checked against the retrieved evidence.

In [6]:
TOP_K = 5

def build_rag_context(question, top_k=TOP_K):
    results = retrieve_bm25(question, top_k=top_k)

    context_parts = []

    for _, row in results.iterrows():
        context_parts.append(
            f"[{row['chunk_id']}] "
            f"{row['document_name']} - Page {row['page']}\n"
            f"{row['text']}"
        )

    context = "\n\n---\n\n".join(context_parts)

    return results, context


# Test using the first Known question
question = test_df.loc[0, "question"]

retrieved_results, context = build_rag_context(question)

print("QUESTION:\n")
print(question)

print("\nRETRIEVED CONTEXT:\n")
print(context)

QUESTION:

Can a rental provider ask me about disputes with a previous landlord when I apply?

RETRIEVED CONTEXT:

[RG_P35] Renters Guide - Page 35
Bond claims
When a rental provider can claim the bond
Your rental provider can claim part or all of the bond for specific things, such as:
• damage caused by you or your visitors (but not fair wear and tear)
• cleaning expenses, if you haven’t left the property reasonably clean.
Read the full list at consumer.vic.gov.au/bondclaims
Process for claiming the bond
Before you move out, you and your rental provider or agent should:
• try to agree on how the bond will be finalised
• set out the agreed division in the bond claim form online.
Your rental provider will usually start the bond claim process online with the
RTBA. Only accept the bond claim if you agree with the amount you will receive
back and all your repayment details are correct. You can ask for changes to the
claim. Once everyone has accepted, the RTBA will usually repay the bond wi

## Generate a Baseline RAG Answer

Pass the retrieved context to a local language model and instruct it to answer only from the supplied evidence. The model must cite the chunk IDs it uses and should state when the retrieved context does not contain enough information.

In [7]:
import requests

OLLAMA_MODEL = "llama3.2:3b"

def generate_rag_answer(question, context):
    prompt = f"""
You are answering questions about Victorian rental rights.

Use ONLY the retrieved context provided below.
Do not use outside knowledge or make assumptions.

Instructions:
- Answer the user's question clearly and concisely.
- Support factual statements using the relevant chunk ID in square brackets, for example [RG_P24].
- Only cite chunks that actually support the statement.
- If the retrieved context does not contain enough information to answer the question, say:
  "The retrieved information does not contain enough information to answer this question."

RETRIEVED CONTEXT:
{context}

QUESTION:
{question}

ANSWER:
"""

    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": OLLAMA_MODEL,
            "prompt": prompt,
            "stream": False,
            "options": {
                "temperature": 0
            }
        }
    )

    response.raise_for_status()

    return response.json()["response"].strip()


answer = generate_rag_answer(question, context)

print("QUESTION:\n")
print(question)

print("\nGENERATED ANSWER:\n")
print(answer)

QUESTION:

Can a rental provider ask me about disputes with a previous landlord when I apply?

GENERATED ANSWER:

The retrieved context does not contain enough information to answer this question.


### Generation Sanity Check

The first baseline response incorrectly refused to answer even though the required evidence was retrieved. To separate retrieval failure from generation failure, the same question is tested using only the known relevant chunk.

In [8]:
# Test generation using only the known relevant chunk

correct_chunk = kb_df[
    kb_df["chunk_id"] == "RG_P07"
].iloc[0]

single_context = (
    f"[{correct_chunk['chunk_id']}] "
    f"{correct_chunk['document_name']} - Page {correct_chunk['page']}\n"
    f"{correct_chunk['text']}"
)

single_chunk_answer = generate_rag_answer(
    question,
    single_context
)

print("QUESTION:\n")
print(question)

print("\nCONTEXT USED:")
print("[RG_P07] only")

print("\nGENERATED ANSWER:\n")
print(single_chunk_answer)

QUESTION:

Can a rental provider ask me about disputes with a previous landlord when I apply?

CONTEXT USED:
[RG_P07] only

GENERATED ANSWER:

The retrieved information does not contain enough information to answer this question.


### Prompt Sanity Check

The model still refused when given only the correct source chunk, suggesting the issue may be the generation prompt rather than retrieval. A simpler prompt is tested below using the same question and evidence.


In [9]:
def generate_simple_answer(question, context):
    prompt = f"""
Answer the question using the information in the context below.

Context:
{context}

Question:
{question}

Give a short answer and cite the supporting chunk ID.
"""

    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": OLLAMA_MODEL,
            "prompt": prompt,
            "stream": False,
            "options": {
                "temperature": 0
            }
        }
    )

    response.raise_for_status()
    return response.json()["response"].strip()


simple_answer = generate_simple_answer(
    question,
    single_context
)

print("GENERATED ANSWER:\n")
print(simple_answer)

GENERATED ANSWER:

No, a rental provider cannot ask you about disputes with a previous landlord when you apply. [RG_P07: The rental provider or agent cannot ask you for any information that is not set out in the form.]


### Grounding and Refusal Check

The simplified prompt successfully answered a Known question, showing that the earlier prompt was causing unnecessary refusals. The prompt is adjusted to still restrict answers to retrieved evidence while only refusing when the context genuinely does not provide an answer.

In [10]:
def generate_grounded_answer(question, context):
    prompt = f"""
Use the retrieved context below to answer the question.

Rules:
- If the context contains information that answers the question, give a short clear answer.
- Base the answer only on the supplied context.
- Cite supporting sources using only their chunk IDs, for example [RG_P24].
- If the context does not provide enough information to answer, say exactly:
  "The retrieved information does not contain enough information to answer this question."

RETRIEVED CONTEXT:
{context}

QUESTION:
{question}

ANSWER:
"""

    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": OLLAMA_MODEL,
            "prompt": prompt,
            "stream": False,
            "options": {
                "temperature": 0
            }
        }
    )

    response.raise_for_status()
    return response.json()["response"].strip()


# Known question - should answer
known_question = test_df.loc[test_df["question_id"] == "K01", "question"].iloc[0]
_, known_context = build_rag_context(known_question)

print("K01:")
print(generate_grounded_answer(known_question, known_context))


# Out-of-KB question - should refuse
oob_question = test_df.loc[test_df["question_id"] == "O01", "question"].iloc[0]
_, oob_context = build_rag_context(oob_question)

print("\nO01:")
print(generate_grounded_answer(oob_question, oob_context))

K01:
The retrieved information does not contain enough information to answer this question.

O01:
The retrieved information does not contain enough information to answer this question.


### Context vs Prompt Check

The model answered correctly when given a simplified prompt and only the relevant chunk. To determine whether the earlier failure was caused by the stricter prompt or by irrelevant retrieved context, the simplified prompt is now tested using the normal top-five BM25 results.

In [11]:
# Test the simple prompt with the normal top-5 context

known_question = test_df.loc[
    test_df["question_id"] == "K01",
    "question"
].iloc[0]

_, known_top5_context = build_rag_context(
    known_question,
    top_k=5
)

top5_simple_answer = generate_simple_answer(
    known_question,
    known_top5_context
)

print("K01 WITH SIMPLE PROMPT + TOP 5 CONTEXT:\n")
print(top5_simple_answer)

K01 WITH SIMPLE PROMPT + TOP 5 CONTEXT:

No, a rental provider cannot ask you about disputes with a previous landlord when you apply. According to the Renters Guide, "The rental provider or agent cannot ask you for the following information in your application: ... whether you’ve taken legal action or had a dispute with a previous rental provider." (Chunk ID: [RG_P07] Renters Guide - Page 7)


### Out-of-KB Sanity Check

The simplified prompt successfully answered a Known question using the normal top-five context. It is now tested on an Out-of-KB question to see whether removing the stricter refusal wording causes unsupported answers.

In [12]:
# Test the simple prompt on an Out-of-KB question

oob_question = test_df.loc[
    test_df["question_id"] == "O01",
    "question"
].iloc[0]

_, oob_top5_context = build_rag_context(
    oob_question,
    top_k=5
)

oob_simple_answer = generate_simple_answer(
    oob_question,
    oob_top5_context
)

print("O01 WITH SIMPLE PROMPT + TOP 5 CONTEXT:\n")
print(oob_simple_answer)

O01 WITH SIMPLE PROMPT + TOP 5 CONTEXT:

No, you cannot sublet your rental property to another person without your rental provider's permission.


### Evidence-Gated Generation

The simple prompt answered supported questions correctly but also produced an unsupported answer for an Out-of-KB question. To reduce this behaviour, generation is separated into an evidence check followed by answer generation. The model must first identify whether the retrieved context contains explicit evidence relevant to the question.

In [13]:
def generate_evidence_gated_answer(question, context):
    evidence_prompt = f"""
Review the retrieved context and question below.

Decide whether the context contains explicit information that can answer the question.

If it does, return:
EVIDENCE_FOUND: YES
EVIDENCE: <copy or briefly identify the relevant information>
SOURCE: <chunk ID>

If it does not, return only:
EVIDENCE_FOUND: NO

RETRIEVED CONTEXT:
{context}

QUESTION:
{question}
"""

    evidence_response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": OLLAMA_MODEL,
            "prompt": evidence_prompt,
            "stream": False,
            "options": {"temperature": 0}
        }
    )

    evidence_response.raise_for_status()
    evidence_check = evidence_response.json()["response"].strip()

    if "EVIDENCE_FOUND: NO" in evidence_check:
        return (
            "The retrieved information does not contain enough information "
            "to answer this question."
        ), evidence_check

    answer_prompt = f"""
Answer the question using only the evidence identified below.

Question:
{question}

Evidence:
{evidence_check}

Give a short, clear answer and cite the supporting chunk ID.
"""

    answer_response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": OLLAMA_MODEL,
            "prompt": answer_prompt,
            "stream": False,
            "options": {"temperature": 0}
        }
    )

    answer_response.raise_for_status()

    return answer_response.json()["response"].strip(), evidence_check


# Test both behaviours
for question_id in ["K01", "O01"]:
    test_question = test_df.loc[
        test_df["question_id"] == question_id,
        "question"
    ].iloc[0]

    _, test_context = build_rag_context(test_question, top_k=5)

    answer, evidence = generate_evidence_gated_answer(
        test_question,
        test_context
    )

    print(f"\n{question_id}")
    print("\nEVIDENCE CHECK:")
    print(evidence)
    print("\nANSWER:")
    print(answer)
    print("-" * 80)


K01

EVIDENCE CHECK:
EVIDENCE_FOUND: YES
EVIDENCE: According to the Renters Guide - Page 7, a rental provider cannot ask you for information about disputes with a previous landlord when you apply.
SOURCE: [RG_P07] Renters Guide - Page 7

ANSWER:
No, a rental provider cannot ask you about disputes with a previous landlord when you apply. (EVIDENCE_FOUND: YES, SOURCE: [RG_P07] Renters Guide - Page 7)
--------------------------------------------------------------------------------

O01

EVIDENCE CHECK:
EVIDENCE_FOUND: YES
EVIDENCE: There is no explicit information in the retrieved context that directly answers the question. However, it can be inferred that subletting a rental property without the landlord's permission is not explicitly prohibited or allowed in the context.
SOURCE: [RG_P29] Renters Guide - Page 29

ANSWER:
No, you cannot sublet your rental property to another person without your rental provider's permission.

Supporting chunk ID: [RG_P29]
---------------------------------

## Generate Baseline Answers

The initial testing showed that the model could answer supported questions from the retrieved evidence, but could also generate answers when the knowledge base did not contain sufficient information. Rather than tuning this behaviour before evaluation, the simple generation prompt is retained as the Version 1 baseline so these weaknesses can be measured across the full test collection.

In [14]:
# Generate baseline answers for the full test collection

baseline_outputs = []

for _, row in test_df.iterrows():
    results, context = build_rag_context(
        row["question"],
        top_k=5
    )

    answer = generate_simple_answer(
        row["question"],
        context
    )

    baseline_outputs.append({
        "question_id": row["question_id"],
        "question_type": row["question_type"],
        "question": row["question"],
        "relevant_chunk_ids": row["relevant_chunk_ids"],
        "expected_answer": row["expected_answer"],
        "retrieved_chunk_ids": results["chunk_id"].tolist(),
        "retrieved_contexts": results["text"].tolist(),
        "generated_answer": answer
    })

baseline_generation_df = pd.DataFrame(baseline_outputs)

display(
    baseline_generation_df[
        [
            "question_id",
            "question_type",
            "question",
            "generated_answer"
        ]
    ]
)

,question_id,question_type,question,generated_answer
0,K01,Known,Can a rental provider ask me about disputes wi...,"No, a rental provider cannot ask you about dis..."
1,K02,Known,Can a rental provider accept an offer above th...,"No, a rental provider cannot accept an offer a..."
2,K03,Known,Does a Victorian rental property need to have ...,"Yes, a Victorian rental property must have a f..."
3,K04,Known,How long do I have to return my completed cond...,You have 5 business days to return your comple...
4,K05,Known,How much rent can I be asked to pay in advance?,You can be asked to pay up to 2 weeks' rent in...
5,K06,Known,How much notice must I receive before my rent ...,You must receive at least 90 days' notice befo...
6,K07,Known,Is a broken toilet considered an urgent repair?,"Yes, a broken toilet is considered an urgent r..."
7,K08,Known,What happens if my rental provider wants to re...,If your rental provider wants to refuse your r...
8,I01,Inferred,My rental has no working fixed heater. Is this...,A non-working fixed heater is considered a non...
9,I02,Inferred,My rent is being increased and I think the new...,You should receive a Notice of rent increase a...


## Review Baseline Generation Outputs

Review each generated answer against the expected answer and retrieved evidence before applying the generation evaluation metrics.

In [15]:
# Print full outputs for manual review

for _, row in baseline_generation_df.iterrows():
    print("=" * 90)
    print(f"{row['question_id']} | {row['question_type']}")
    
    print("\nQUESTION:")
    print(row["question"])
    
    print("\nEXPECTED ANSWER:")
    print(row["expected_answer"])
    
    print("\nRETRIEVED CHUNKS:")
    print(row["retrieved_chunk_ids"])
    
    print("\nGENERATED ANSWER:")
    print(row["generated_answer"])
    
    print()

K01 | Known

QUESTION:
Can a rental provider ask me about disputes with a previous landlord when I apply?

EXPECTED ANSWER:
No. A rental provider or agent cannot ask whether you have taken legal action or had a dispute with a previous rental provider.

RETRIEVED CHUNKS:
['RG_P35', 'RG_P38', 'RG_P04', 'RG_P23', 'RG_P07']

GENERATED ANSWER:
No, a rental provider cannot ask you about disputes with a previous landlord when you apply. According to [RG_P07] Renters Guide - Page 7, a rental provider cannot ask you for information that is not set out in the prescribed rental application form, specifically:

"...whether you’ve taken legal action or had a dispute with a previous rental provider."

This is stated in the section "Before you apply" under "Documents and information you may need to provide:".

K02 | Known

QUESTION:
Can a rental provider accept an offer above the advertised rent?

EXPECTED ANSWER:
No. Rental providers and agents cannot ask for, invite or accept offers of rent higher 

## Prepare Manual Generation Evaluation

The baseline answers are evaluated separately for correctness, completeness, faithfulness, citation accuracy and Out-of-KB abstention. Keeping these measures separate helps distinguish different failure types instead of reducing answer quality to a single score.

In [16]:
# Create a manual evaluation table

manual_eval_df = baseline_generation_df[
    [
        "question_id",
        "question_type",
        "question",
        "expected_answer",
        "retrieved_chunk_ids",
        "generated_answer"
    ]
].copy()

# Blank fields for manual review
manual_eval_df["correct"] = pd.NA
manual_eval_df["complete"] = pd.NA
manual_eval_df["faithful"] = pd.NA
manual_eval_df["citation_correct"] = pd.NA
manual_eval_df["correct_abstention"] = pd.NA
manual_eval_df["review_notes"] = ""

display(manual_eval_df)

,question_id,question_type,question,expected_answer,retrieved_chunk_ids,generated_answer,correct,complete,faithful,citation_correct,correct_abstention,review_notes
0,K01,Known,Can a rental provider ask me about disputes wi...,No. A rental provider or agent cannot ask whet...,"[RG_P35, RG_P38, RG_P04, RG_P23, RG_P07]","No, a rental provider cannot ask you about dis...",<NA>,<NA>,<NA>,<NA>,<NA>,
1,K02,Known,Can a rental provider accept an offer above th...,No. Rental providers and agents cannot ask for...,"[RG_P08, RG_P21, MS_P01, RG_P12, RG_P13]","No, a rental provider cannot accept an offer a...",<NA>,<NA>,<NA>,<NA>,<NA>,
2,K03,Known,Does a Victorian rental property need to have ...,Yes. Rental properties must have a fixed heate...,"[RG_P13, MS_P01, RG_P39, RG_P17, RG_P04]","Yes, a Victorian rental property must have a f...",<NA>,<NA>,<NA>,<NA>,<NA>,
3,K04,Known,How long do I have to return my completed cond...,You must return one completed and signed copy ...,"[RG_P23, RG_P17, RG_P19, RG_P34, RG_P36]",You have 5 business days to return your comple...,<NA>,<NA>,<NA>,<NA>,<NA>,
4,K05,Known,How much rent can I be asked to pay in advance?,"If rent is paid weekly, you can be asked for u...","[RG_P21, RG_P22, RG_P17, RG_P23, RG_P08]",You can be asked to pay up to 2 weeks' rent in...,<NA>,<NA>,<NA>,<NA>,<NA>,
5,K06,Known,How much notice must I receive before my rent ...,The rental provider must give you a Notice of ...,"[RG_P23, RG_P22, RG_P17, RG_P24, RG_P18]",You must receive at least 90 days' notice befo...,<NA>,<NA>,<NA>,<NA>,<NA>,
6,K07,Known,Is a broken toilet considered an urgent repair?,Yes. A blocked or broken toilet system is lega...,"[RG_P24, RG_P25, RG_P26, RG_P15, MS_P01]","Yes, a broken toilet is considered an urgent r...",<NA>,<NA>,<NA>,<NA>,<NA>,
7,K08,Known,What happens if my rental provider wants to re...,If the rental provider wants to refuse consent...,"[RG_P28, RG_P23, RG_P22, RG_P29, RG_P34]",If your rental provider wants to refuse your r...,<NA>,<NA>,<NA>,<NA>,<NA>,
8,I01,Inferred,My rental has no working fixed heater. Is this...,Yes. A rental property must have a working fix...,"[RG_P25, RG_P26, MS_P01, RG_P23, RG_P13]",A non-working fixed heater is considered a non...,<NA>,<NA>,<NA>,<NA>,<NA>,
9,I02,Inferred,My rent is being increased and I think the new...,You must receive a Notice of rent increase at ...,"[RG_P23, RG_P22, RG_P17, RG_P18, RG_P25]",You should receive a Notice of rent increase a...,<NA>,<NA>,<NA>,<NA>,<NA>,


In [17]:
# Add citation presence as a separate measure
if "citation_present" not in manual_eval_df.columns:
    citation_pos = manual_eval_df.columns.get_loc("citation_correct")
    manual_eval_df.insert(citation_pos, "citation_present", pd.NA)

known_scores = {
    "K01": [1, 1, 1, 1, 1, "Correct and fully supported."],
    "K02": [1, 1, 1, 1, 1, "Correct and fully supported."],
    "K03": [1, 0, 1, 1, 0, "Correct core answer, but misses energy-efficiency requirement and cites the wrong chunk."],
    "K04": [1, 1, 1, 1, 1, "Correct answer and citation."],
    "K05": [1, 0, 1, 0, pd.NA, "Correct main limits but omits the rule for weekly rent above $900. No citation provided."],
    "K06": [1, 1, 1, 1, 0, "Correct answer, but cited chunk does not support the 90-day notice claim."],
    "K07": [1, 1, 1, 1, 1, "Correct and supported by RG_P24."],
    "K08": [1, 1, 1, 1, 1, "Correct and fully supported."]
}

for question_id, scores in known_scores.items():
    mask = manual_eval_df["question_id"] == question_id

    manual_eval_df.loc[mask, [
        "correct",
        "complete",
        "faithful",
        "citation_present",
        "citation_correct",
        "review_notes"
    ]] = scores

display(
    manual_eval_df[
        manual_eval_df["question_type"] == "Known"
    ][
        [
            "question_id",
            "correct",
            "complete",
            "faithful",
            "citation_present",
            "citation_correct",
            "review_notes"
        ]
    ]
)

,question_id,correct,complete,faithful,citation_present,citation_correct,review_notes
0,K01,1,1,1,1,1,Correct and fully supported.
1,K02,1,1,1,1,1,Correct and fully supported.
2,K03,1,0,1,1,0,"Correct core answer, but misses energy-efficie..."
3,K04,1,1,1,1,1,Correct answer and citation.
4,K05,1,0,1,0,<NA>,Correct main limits but omits the rule for wee...
5,K06,1,1,1,1,0,"Correct answer, but cited chunk does not suppo..."
6,K07,1,1,1,1,1,Correct and supported by RG_P24.
7,K08,1,1,1,1,1,Correct and fully supported.


In [18]:
inferred_scores = {
    "I01": [
        0, 0, 0, 0, pd.NA,
        "Incorrectly classifies the heater issue as non-urgent. The retrieved results also missed RG_P24, which explicitly links minimum-standard failures to urgent repairs."
    ],
    
    "I02": [
        1, 0, 1, 0, pd.NA,
        "Correct notice period and rent-assessment process, but omits the RDRV or VCAT escalation step. No citation provided."
    ],
    
    "I03": [
        1, 0, 1, 1, 1,
        "Correctly identifies the condition report and photos as evidence, but omits the exit report comparison, fair wear and tear rule, and RDRV dispute process."
    ],
    
    "I04": [
        1, 1, 1, 1, 1,
        "Correctly explains protection against retaliatory eviction and the usual notice requirements. RG_P32 supports the notice requirements, although the first claim is supported by RG_P33."
    ]
}

for question_id, scores in inferred_scores.items():
    mask = manual_eval_df["question_id"] == question_id

    manual_eval_df.loc[mask, [
        "correct",
        "complete",
        "faithful",
        "citation_present",
        "citation_correct",
        "review_notes"
    ]] = scores

display(
    manual_eval_df[
        manual_eval_df["question_type"] == "Inferred"
    ][
        [
            "question_id",
            "correct",
            "complete",
            "faithful",
            "citation_present",
            "citation_correct",
            "review_notes"
        ]
    ]
)

,question_id,correct,complete,faithful,citation_present,citation_correct,review_notes
8,I01,0,0,0,0,<NA>,Incorrectly classifies the heater issue as non...
9,I02,1,0,1,0,<NA>,Correct notice period and rent-assessment proc...
10,I03,1,0,1,1,1,Correctly identifies the condition report and ...
11,I04,1,1,1,1,1,Correctly explains protection against retaliat...


In [19]:
out_of_kb_scores = {
    "O01": [
        0, 0, 0, 0, pd.NA, 0,
        "Incorrectly answers an Out-of-KB question instead of abstaining. The claim is not supported by the retrieved context."
    ],

    "O02": [
        1, 1, 1, 0, pd.NA, 1,
        "Correctly recognises that the retrieved context does not contain the requested information."
    ],

    "O03": [
        0, 0, 0, 0, pd.NA, 0,
        "Incorrectly answers an Out-of-KB question instead of abstaining. The claim about security cameras is unsupported by the retrieved context."
    ]
}

for question_id, scores in out_of_kb_scores.items():
    mask = manual_eval_df["question_id"] == question_id

    manual_eval_df.loc[mask, [
        "correct",
        "complete",
        "faithful",
        "citation_present",
        "citation_correct",
        "correct_abstention",
        "review_notes"
    ]] = scores

display(
    manual_eval_df[
        manual_eval_df["question_type"] == "Out-of-KB"
    ][
        [
            "question_id",
            "correct",
            "complete",
            "faithful",
            "citation_present",
            "citation_correct",
            "correct_abstention",
            "review_notes"
        ]
    ]
)

,question_id,correct,complete,faithful,citation_present,citation_correct,correct_abstention,review_notes
12,O01,0,0,0,0,<NA>,0,Incorrectly answers an Out-of-KB question inst...
13,O02,1,1,1,0,<NA>,1,Correctly recognises that the retrieved contex...
14,O03,0,0,0,0,<NA>,0,Incorrectly answers an Out-of-KB question inst...


## Summarise Baseline Generation Performance

Summarise the manual evaluation results overall and by question type. Citation accuracy is calculated only for answers that included a citation, while abstention accuracy is evaluated only for Out-of-KB questions.

In [21]:
# Main generation metrics by question type

generation_summary = (
    manual_eval_df
    .groupby("question_type")[
        ["correct", "complete", "faithful", "citation_present"]
    ]
    .mean()
    .astype(float)
    .round(3)
)

# Citation correctness only where a citation was provided
citation_summary = (
    manual_eval_df[
        manual_eval_df["citation_present"] == 1
    ]
    .groupby("question_type")["citation_correct"]
    .mean()
    .astype(float)
    .round(3)
)

generation_summary["citation_correct"] = citation_summary

display(generation_summary)


# Overall baseline
overall_generation = pd.DataFrame({
    "correct": [manual_eval_df["correct"].mean()],
    "complete": [manual_eval_df["complete"].mean()],
    "faithful": [manual_eval_df["faithful"].mean()],
    "citation_present": [manual_eval_df["citation_present"].mean()],
    "citation_correct": [
        manual_eval_df.loc[
            manual_eval_df["citation_present"] == 1,
            "citation_correct"
        ].mean()
    ],
    "out_of_kb_abstention": [
        manual_eval_df.loc[
            manual_eval_df["question_type"] == "Out-of-KB",
            "correct_abstention"
        ].mean()
    ]
}, index=["Overall"]).astype(float).round(3)

display(overall_generation)

,correct,complete,faithful,citation_present,citation_correct
question_type,,,,,
Inferred,0.750,0.250,0.750,0.500,1.000
Known,1.000,0.750,1.000,0.875,0.714
Out-of-KB,0.333,0.333,0.333,0.000,NaN


,correct,complete,faithful,citation_present,citation_correct,out_of_kb_abstention
Overall,0.8,0.533,0.8,0.6,0.778,0.333


## Baseline RAG Evaluation Summary

The Version 1 baseline was built to give us a clear starting point before making any improvements to the RAG system. We evaluated the pipeline in two separate stages: **retrieval** and **answer generation**. This helps us understand whether a poor final answer comes from failing to retrieve the right information or from the language model using the retrieved information incorrectly.

### Retrieval

We first built a BM25 keyword-based retriever over the 32 chunks in the Version 1 rental-rights knowledge base. Retrieval was evaluated on the Known and Inferred questions using **Recall, MRR and NDCG**.

At `top_k = 5`, the baseline achieved:

- **Recall@5:** 0.979
- **MRR@5:** 0.808
- **NDCG@5:** 0.831

Known questions achieved perfect Recall@5, while Inferred questions achieved 0.938. This shows that BM25 was generally very good at finding relevant information, although the correct evidence was not always ranked near the top.

We also compared different retrieval depths. Recall increased from **0.806 at k=3 to 0.979 at k=5**, while increasing to 10 chunks only improved it slightly further to 1.000. Because the extra five chunks would add considerably more context for only a small retrieval improvement, `top_k = 5` was selected for the baseline generation pipeline.

### Answer Generation

The five retrieved chunks were then passed to the local `llama3.2:3b` model. Early testing showed an important prompt trade-off. A strict grounding prompt caused the model to refuse questions even when the exact answer was present in the retrieved context. A simpler prompt successfully answered these supported questions, but was more willing to generate answers when the knowledge base did not contain the required information.

Rather than continuing to tune the prompt before evaluation, the simpler prompt was kept as the Version 1 baseline so these weaknesses could be measured properly.

The 15 generated answers were manually evaluated for **correctness, completeness, faithfulness, citation behaviour and Out-of-KB abstention**.

Overall results were:

- **Correct answers:** 0.800
- **Complete answers:** 0.533
- **Faithful answers:** 0.800
- **Answers containing a citation:** 0.600
- **Correct citations when provided:** 0.778
- **Correct Out-of-KB abstention:** 0.333

### What We Observed

The Known questions performed best. All eight produced a broadly correct answer, showing that when the required information is available, the model can generally make good use of the retrieved evidence. However, some answers were incomplete or cited the wrong chunk. For example, the fixed-heater answer missed the additional energy-efficiency requirement, while another answer gave the correct 90-day rent increase notice but cited an unrelated retrieved chunk.

The Inferred questions were more difficult because they required information to be combined across multiple chunks. Some answers captured only part of the expected response. The clearest failure was the heater repair question, where the model incorrectly stated that a non-working fixed heater was a **non-urgent repair**. BM25 had failed to retrieve `RG_P24`, which contained the important connection between minimum-standard failures and urgent repairs. This gives us a useful example of how a retrieval weakness can flow through into an incorrect generated answer.

The largest weakness was **Out-of-KB behaviour**. Only one of the three unsupported questions was correctly rejected. For the subletting and security-camera questions, the model gave confident answers even though the retrieved context did not contain evidence supporting those claims. This is a clear hallucination and grounding problem.

### What This Means for the Next Steps

The baseline shows that basic BM25 retrieval is already reasonably strong, so future improvements should not focus only on increasing retrieval scores. The more important issues are improving retrieval for multi-part Inferred questions, making generated answers more complete, improving source attribution, and preventing the model from answering when there is not enough evidence.

The next versions of the system can therefore be compared against this fixed baseline. Improvements could include expanding the knowledge base with the additional rental guidelines, testing stronger or semantic retrieval approaches, improving how multiple pieces of evidence are selected, and introducing a more reliable method for deciding when the system should abstain. The same test collection and evaluation measures can then be reused to determine whether these changes actually improve the system rather than relying on individual examples.
